In [ ]:
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

In [ ]:
# Load model + processor (force CPU, use local cache for reliability)
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32", cache_dir="./saved_models")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", cache_dir="./saved_models")

# Alternative: explicit CPU placement (useful if CUDA is available but you want CPU)
# model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32", cache_dir="./saved_models").to("cpu")
# processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", cache_dir="./saved_models")

SSLError: (MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /openai/clip-vit-base-patch32/resolve/main/config.json (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)')))"), '(Request ID: db24c48a-a0f4-430c-9c06-f17e70cd258d)')

In [ ]:
# Load your image (can be any format)
image = Image.open(r"..\data\Chest X-Rays\val\COVID19\COVID19(568).jpg").convert("RGB")

# Define candidate labels (text descriptions)
text_descriptions = [
    "a chest X-ray",
    "a CT scan",
    "a natural photograph",
    "a painting",
    "a selfie",
    "a dog",
    "a screenshot"
]

# Prepare inputs
inputs = processor(text=text_descriptions, images=image, return_tensors="pt", padding=True)

In [ ]:
# Run model
outputs = model(**inputs)
logits_per_image = outputs.logits_per_image  # shape: [1, N_texts]
probs = logits_per_image.softmax(dim=1)      # softmax over text options

# Get top match
top_index = torch.argmax(probs)
top_label = text_descriptions[top_index]

print(f"Top label: {top_label}")
print(f"Confidence: {probs[0, top_index].item():.4f}")

Top label: a chest X-ray
Confidence: 0.9951
